# Lab: Post-Quantum Cryptography for Blockchain

**Student name:** ____________________  **Date:** ____________________

In this lab you will:
- Understand why quantum computers threaten blockchain cryptography
- Implement a lattice-based encryption scheme (LWE) from scratch
- Encrypt and decrypt a blockchain transaction
- Reflect on why lattice cryptography is quantum-resistant

Use only basic Python — no NumPy, no external libraries.


## Part 1: The Quantum Threat to Blockchain

Modern blockchains use **Elliptic Curve Cryptography (ECC)** for digital signatures.
The security of ECC rests on the **discrete logarithm problem**: given $g$ and $g^x$, find $x$.

**Shor's algorithm** (1994) can solve this in polynomial time on a quantum computer using
the **Quantum Fourier Transform (QFT)**. This means a large enough quantum computer could:
- Recover private keys from public keys
- Forge any signature
- Completely break blockchain security

The code below demonstrates the **scaling difference** between classical and quantum factoring.
Read it, run it, then answer the question below.


In [ ]:
# Run this cell to see how classical vs quantum factoring scales
import math

print("Scaling of classical vs quantum factoring:")
print(f"{'Key size':<12} {'Classical steps':<20} {'Quantum steps':<20}")
print("-" * 52)
for bits in [16, 32, 64, 128, 256]:
    classical_steps = 2 ** (bits // 2)
    quantum_steps = bits ** 3
    print(f"{bits}-bit          {classical_steps:<20} {quantum_steps:<20}")


**Question 1 (written answer):** Why does the quantum speedup matter so much for blockchain?
What happens to the security assumptions of ECDSA if Shor's algorithm runs efficiently?

Write your answer in 2-3 sentences.

→ *Your answer:*





## Part 2: Lattice-Based Cryptography

Lattice problems are believed to be **hard for both classical and quantum computers**
because they lack the hidden algebraic structure that Shor's algorithm exploits.

The **Learning With Errors (LWE)** problem:
- Given a matrix $A$ and $\mathbf{b} = A\mathbf{s} + \mathbf{e}$ where $\mathbf{e}$ is a small error
- It is hard to recover $\mathbf{s}$

We use this to build encryption:
- **KeyGen** → $(A, \mathbf{b} = A\mathbf{s} + \mathbf{e})$  (public key), $\mathbf{s}$ (secret key)
- **Encrypt** → $\mathbf{u} = A^T\mathbf{r}$, $v = \mathbf{b}\cdot\mathbf{r} + m\lfloor q/2 \rfloor$
- **Decrypt** → $d = v - \mathbf{s}\cdot\mathbf{u} = \mathbf{e}\cdot\mathbf{r} + m\lfloor q/2 \rfloor$, then decode the bit


## Part 3: Implement the Helper Functions

Complete the functions below. Each one is a short utility used by the encryption scheme.
Run the test cell after each function to check your work.


In [ ]:
# TASK 3a: Implement dot product
# Return the sum of a[i] * b[i] for all i.
# Example: dot([1,2,3], [4,5,6]) = 1*4 + 2*5 + 3*6 = 32

def dot(a, b):
    # TODO: write your code here
    # Hint: use a loop or a list comprehension with sum()
    pass

# --- Test your dot function ---
result = dot([1, 2, 3], [4, 5, 6])
print(f"dot([1,2,3], [4,5,6]) = {result}")
assert result == 32, f"Expected 32, got {result}"
print("dot() test passed!")


In [ ]:
# TASK 3b: Implement matrix-vector multiplication (mod q)
# Multiply matrix A (m rows, n columns) by vector v (length n).
# Result[i] = (sum of A[i][j] * v[j] for all j) mod q
#
# Example:
#   A = [[1, 2], [3, 4]],  v = [5, 6],  q = 20
#   result[0] = (1*5 + 2*6) % 20 = 17 % 20 = 17
#   result[1] = (3*5 + 4*6) % 20 = 39 % 20 = 19

def mat_vec_mul(A, v, q):
    # TODO: write your code here
    # Hint: for each row i in A, compute dot(row, v) % q
    pass

# --- Test your mat_vec_mul ---
A_test = [[1, 2], [3, 4]]
v_test = [5, 6]
result = mat_vec_mul(A_test, v_test, 20)
print(f"mat_vec_mul([[1,2],[3,4]], [5,6], 20) = {result}")
assert result == [17, 19], f"Expected [17, 19], got {result}"
print("mat_vec_mul() test passed!")


In [ ]:
# TASK 3c: Implement small error vector generator
# Return a list of length n where each entry is randomly -1, 0, or 1.
# Use random.choice([-1, 0, 1]) for each entry.

import random

def small_vector(n):
    # TODO: write your code here
    pass

# --- Test your small_vector ---
sv = small_vector(10)
print(f"small_vector(10) = {sv}")
assert len(sv) == 10, f"Expected length 10, got {len(sv)}"
assert all(x in [-1, 0, 1] for x in sv), f"Entries must be -1, 0, or 1"
print("small_vector() test passed!")


In [ ]:
# TASK 3d: Implement uniform random vector
# Return a list of length n where each entry is random in [0, q).
# Use random.randrange(0, q) for each entry.
# If q is None, return random bits (0 or 1) using random.randint(0, 1)

def random_vector(n, q=None):
    # TODO: write your code here
    pass

# --- Test your random_vector ---
rv = random_vector(5, 10)
print(f"random_vector(5, 10) = {rv}")
assert len(rv) == 5
assert all(0 <= x < 10 for x in rv)
print("random_vector() with q test passed!")

bv = random_vector(8)
print(f"random_vector(8) = {bv}")
assert all(x in [0, 1] for x in bv)
print("random_vector() without q test passed!")


In [ ]:
# TASK 3e: Implement matrix transpose
# Given an m×n matrix A, return the n×m transpose A^T
# where result[j][i] = A[i][j]
#
# Example: transpose([[1,2],[3,4],[5,6]]) = [[1,3,5],[2,4,6]]

def transpose(A):
    # TODO: write your code here
    # Hint: get m = len(A), n = len(A[0]), then build result
    pass

# --- Test your transpose ---
A_test = [[1, 2], [3, 4], [5, 6]]
result = transpose(A_test)
print(f"transpose of {A_test} = {result}")
assert result == [[1, 3, 5], [2, 4, 6]], f"Expected [[1,3,5],[2,4,6]], got {result}"
print("transpose() test passed!")


In [ ]:
# TASK 3f: Implement random matrix
# Return an m×n matrix where each entry is random in [0, q).
# Each row should be a random_vector(n, q).

def random_matrix(m, n, q):
    # TODO: write your code here (use your random_vector helper)
    pass

# --- Test your random_matrix ---
M = random_matrix(3, 4, 10)
print(f"random_matrix(3, 4, 10) = {M}")
assert len(M) == 3
assert len(M[0]) == 4
print("random_matrix() test passed!")


## Part 4: Key Generation

Now implement the LWE key generation.

Recall: public key = $(A, \mathbf{b})$ where $\mathbf{b} = A\mathbf{s} + \mathbf{e} \pmod{q}$,
and the secret key is $\mathbf{s}$.


In [ ]:
# TASK 4: Implement LWE key generation
# Parameters (these are small for demonstration — real LWE uses n > 512, q > 2^12):
n = 8      # dimension
q = 97     # modulus (must be >> n for correct decryption)

def keygen():
    """
    Generate an LWE key pair.
    Returns: ((A, b), s)
      - A: n×n random matrix
      - s: secret vector (uniform random)
      - e: small error vector
      - b: (A*s + e) mod q  (this is the "noisy" public vector)
    """
    # Step 1: Generate a random n×n matrix A with entries mod q
    A = None  # TODO: use random_matrix()

    # Step 2: Generate a random secret vector s of length n with entries mod q
    s = None  # TODO: use random_vector()

    # Step 3: Generate a small error vector e of length n (entries -1, 0, 1)
    e = None  # TODO: use small_vector()

    # Step 4: Compute b = A*s + e (mod q)
    # Use mat_vec_mul for A*s, then add e element-wise, then mod q
    b = None  # TODO: write your code here

    return (A, b), s

# --- Generate and inspect a key pair ---
(pk, sk) = keygen()
A, b = pk
print(f"n = {n}, q = {q}")
print(f"Secret key s = {sk}")
print(f"Public key b = {b}")
print("Key generation completed! (Run the next cell to verify)" if all(v is not None for v in [A, b, sk]) else "Still needs work...")


**Question 2 (written answer):** Look at the values of $\mathbf{b}$ and $\mathbf{s}$ printed above.
The public key $\mathbf{b}$ is a "noisy" version of $A\mathbf{s}$. Why does adding the small
error $\mathbf{e}$ make it hard for an attacker to recover $\mathbf{s}$ from $(A, \mathbf{b})$?
What would happen if $\mathbf{e} = \mathbf{0}$ (all zeros)?

→ *Your answer:*





## Part 5: Encryption

Encrypt a single bit $m \in \{0, 1\}$:
- Choose a random small vector $\mathbf{r}$
- Compute $\mathbf{u} = A^T \mathbf{r} \pmod{q}$
- Compute $v = \mathbf{b} \cdot \mathbf{r} + m \lfloor q/2 \rfloor \pmod{q}$
- Return ciphertext $(\mathbf{u}, v)$


In [ ]:
# TASK 5: Implement LWE encryption
# The global variables n, q and the helper functions from above are available.

def encrypt(pk, bit):
    """
    Encrypt a single bit (0 or 1) using the LWE public key.
    pk = (A, b)
    Returns: (u, v)  — the ciphertext
    """
    A, b = pk

    # Step 1: Choose a random small vector r (entries -1, 0, 1)
    r = None  # TODO: use small_vector()

    # Step 2: Compute u = A^T * r (mod q)
    # Hint: transpose(A) then mat_vec_mul
    u = None  # TODO: write your code here

    # Step 3: Compute v = (b·r + bit * (q//2)) % q
    # Hint: use your dot() function, then add bit * (q // 2), then mod q
    v = None  # TODO: write your code here

    return u, v

# --- Test encryption ---
ct0 = encrypt(pk, 0)
ct1 = encrypt(pk, 1)
print(f"Encrypted bit 0: u (first 3) = {ct0[0][:3]}..., v = {ct0[1]}")
print(f"Encrypted bit 1: u (first 3) = {ct1[0][:3]}..., v = {ct1[1]}")
print("Encryption works!" if ct0[0] and ct1[0] else "Still needs work...")


## Part 6: Decryption

Decrypt a ciphertext $(\mathbf{u}, v)$:
- Compute $d = v - \mathbf{s} \cdot \mathbf{u} \pmod{q}$
- If $d$ is close to $\lfloor q/2 \rfloor$, the bit is **1**
- If $d$ is close to $0$, the bit is **0**

Why does this work? Let's trace the math:

$d = v - \mathbf{s} \cdot \mathbf{u}$
$= (\mathbf{b} \cdot \mathbf{r} + m\lfloor q/2 \rfloor) - \mathbf{s} \cdot (A^T \mathbf{r})$
$= ((A\mathbf{s} + \mathbf{e}) \cdot \mathbf{r} + m\lfloor q/2 \rfloor) - (A\mathbf{s}) \cdot \mathbf{r}$
$= \mathbf{e} \cdot \mathbf{r} + m\lfloor q/2 \rfloor$

The error term $\mathbf{e} \cdot \mathbf{r}$ is small (bounded by $n$). Since $\lfloor q/2 \rfloor \gg n$,
we can clearly distinguish $m=0$ from $m=1$.


In [ ]:
# TASK 6: Implement LWE decryption
# The global variables n, q are still available.

def decrypt(sk, ct):
    """
    Decrypt a ciphertext (u, v) using the LWE secret key.
    Returns: 0 or 1
    """
    u, v = ct

    # Step 1: Compute d = (v - dot(s, u)) % q
    d = None  # TODO: write your code here

    # Step 2: Decide the bit based on whether d is near q//2 or near 0
    # If d > q//4 and d < 3*q//4, it is closer to q//2 -> return 1
    # Otherwise it is closer to 0 -> return 0
    threshold = q // 4
    # TODO: write the if-else logic and return the correct bit
    pass

# --- Test decryption ---
d0 = decrypt(sk, ct0)
d1 = decrypt(sk, ct1)
print(f"Decrypt(encrypt(0)) = {d0}")
print(f"Decrypt(encrypt(1)) = {d1}")
assert d0 == 0, f"Expected 0, got {d0}"
assert d1 == 1, f"Expected 1, got {d1}"
print("Decryption works correctly!" if d0 == 0 and d1 == 1 else "Something is wrong...")


In [ ]:
# Run this cell to test encryption + decryption of multiple bits
print(f"Testing LWE encryption with n={n}, q={q}")
print("-" * 45)
all_pass = True
for bit in [0, 1, 0, 1, 1, 0, 1]:
    ct = encrypt(pk, bit)
    decrypted = decrypt(sk, ct)
    status = "OK" if decrypted == bit else "FAIL"
    if decrypted != bit:
        all_pass = False
    print(f"  Original: {bit}  ->  Decrypted: {decrypted}  [{status}]")

if all_pass:
    print("\nSUCCESS: LWE encryption and decryption work correctly!")
else:
    print("\nSome tests failed. Check your implementation.")


**Question 3 (written answer):** In the derivation above, the error term $\mathbf{e} \cdot \mathbf{r}$
cancels out during decryption for the legitimate user but remains unknown to the attacker.
Explain in your own words why an attacker who sees $(A, \mathbf{b})$ and $(\mathbf{u}, v)$
cannot recover the message bit without knowing $\mathbf{s}$.

→ *Your answer:*





## Part 7: Encrypt a Blockchain Transaction

Now we apply our LWE encryption to a real use case: encrypting a blockchain transaction.
Alice wants to send Bob a transaction, encrypted so only Bob (who knows the secret key)
can read it. This replaces the ECDH key agreement used in many blockchain protocols.

First we need to convert a string (the transaction) into bits, and back.


In [ ]:
# TASK 7a: Implement bit conversion helpers

def message_to_bits(msg):
    """Convert a string to a list of bits (0s and 1s)."""
    bits = []
    for char in msg.encode('utf-8'):
        # TODO: for each character, extract its 8 bits
        # Hint: use a loop from 0 to 7, shift and mask:
        #   (char >> (7 - i)) & 1
        pass
    return bits

def bits_to_message(bits):
    """Convert a list of bits back to a string."""
    chars = []
    for i in range(0, len(bits), 8):
        byte = 0
        # TODO: reconstruct each byte from 8 bits
        # Hint: byte = (byte << 1) | bits[i + j]
        pass
    return bytes(chars).decode('utf-8')

# --- Test your conversion helpers ---
test_msg = "Hello!"
bits = message_to_bits(test_msg)
recovered = bits_to_message(bits)
print(f"Original: {test_msg}")
print(f"Bits:     {bits}")
print(f"Recovered: {recovered}")
assert recovered == test_msg, f"Roundtrip failed: got '{recovered}'"
print("Bit conversion works correctly!")


In [ ]:
# TASK 7b: Encrypt and decrypt a blockchain transaction

# Alice's transaction
transaction = '{"from": "Alice", "to": "Bob", "amount": 5.0}'
print(f"Original transaction: {transaction}\n")

# Generate Alice's key pair (her wallet)
(alice_pk, alice_sk) = keygen()

# Convert transaction to bits, then encrypt each bit
tx_bits = message_to_bits(transaction)
print(f"Transaction is {len(tx_bits)} bits")

# TODO: Encrypt each bit using encrypt(alice_pk, bit)
ciphertext = None  # Hint: use a list comprehension

print(f"Encrypted into {len(ciphertext)} ciphertext tuples")

# TODO: Decrypt each ciphertext tuple using decrypt(alice_sk, ct)
decrypted_bits = None  # Hint: use a list comprehension

# Convert bits back to text
decrypted_tx = bits_to_message(decrypted_bits)
print(f"Decrypted transaction: {decrypted_tx}")
print(f"Match: {decrypted_tx == transaction}")

if decrypted_tx == transaction:
    print("\nSUCCESS: Blockchain transaction encrypted and decrypted with lattice crypto!")
else:
    print("\nSomething went wrong. Check your encrypt/decrypt or bit conversion.")


## Part 8: Current Standards (Reading)

NIST has standardized the following Post-Quantum Cryptography algorithms:

| Algorithm | Type | FIPS Standard | Replaces |
|-----------|------|---------------|----------|
| **ML-KEM (Kyber)** | Key Encapsulation | FIPS 203 | RSA-KEM, ECDH |
| **ML-DSA (Dilithium)** | Digital Signatures | FIPS 204 | ECDSA, EdDSA |
| **SLH-DSA (SPHINCS+)** | Hash-based Signatures | FIPS 205 | Conservative fallback |

Google is migrating its infrastructure to PQC by 2026, and Chrome already uses
hybrid X25519+Kyber key agreement. These same algorithms will eventually replace
ECDSA and ECDH in blockchain protocols.


**Question 4 (written answer):** The scheme you implemented in this lab is a simplified
version of the same mathematics that powers CRYSTALS-Kyber (ML-KEM) and CRYSTALS-Dilithium (ML-DSA).
Why is lattice-based cryptography considered a good replacement for ECC in blockchain?
What advantage does it have over other post-quantum approaches?

→ *Your answer:*





## Congratulations!

You have implemented a **Learning With Errors (LWE)** encryption scheme — the same
mathematical foundation used by NIST's post-quantum cryptography standards.

### What you accomplished:
1. Learned why quantum computers break ECC and RSA
2. Implemented lattice helper functions (dot product, matrix-vector multiply, transpose)
3. Built LWE key generation, encryption, and decryption
4. Encrypted a blockchain transaction using lattice-based cryptography
5. Understood why lattice problems resist quantum attacks

### Before submitting:
- Make sure all code cells run without errors
- Write your answers to Questions 1-4 in the markdown cells above
- Save the notebook and submit
